# Chapter 33 – Worked examples

This notebook runs the examples of [Chapter 33](../index.md) live. The code lives in the modules next to it
(`dcgrid.py`, `planted.py`, `recompile.py`, `bridge_debug.py`, `pydantic_extras.py`, `bsdf_property.py` and the
property test `test_lodf_property.py`); the cells only call them.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import bridge_debug
import bsdf_property
import dcgrid
import numpy as np
import planted
import pydantic_extras
import recompile
import test_lodf_property
from hypothesis import find, settings

## The property test of the worked example

`test_lodf_matches_resolve` states that N-1 flows from LODFs equal a DC re-solve without the outaged branch, on
random meshed grids. Calling the decorated function runs Hypothesis' search (200 examples); silence means no
counterexample.

In [ ]:
test_lodf_property.test_lodf_matches_resolve()
print('200 random meshed grids, no counterexample')

Delete the diagonal fix (`fix_diagonal=False` in `dcgrid.lodf_matrix`) and Hypothesis finds a grid at once; the
bare formula gives $\mathrm{LODF}_{b,b} = c_b/(1-c_b) \neq -1$. `find` returns the shrunk counterexample.

In [ ]:
grid = find(
    test_lodf_property.meshed_grids(),
    lambda g: dcgrid.n1_mismatch(g[0], g[1], 1.0 / g[2], g[3], fix_diagonal=False) > 1e-6,
    settings=settings(max_examples=200, database=None),
)
fr, to, x, p = grid
print('from', fr, 'to', to, 'x', x, 'p', p)
print('mismatch with the fix   :', dcgrid.n1_mismatch(fr, to, 1.0 / x, p))
print('mismatch without the fix:', dcgrid.n1_mismatch(fr, to, 1.0 / x, p, fix_diagonal=False))

## Three planted bugs on IEEE 14

`planted.symptoms` runs six checks against pandapower's DC power flow and against a re-solve. Each bug fails the
signed oracle, and each bug also has a check that misses it: that is why one kind of test is never enough.

In [ ]:
data = planted.ieee14()
tables = {bug: planted.symptoms(data, bug) for bug in (None, *planted.BUGS)}
print(f'{"check":42s}' + ''.join(f'{b!s:>13s}' for b in ('correct', *planted.BUGS)))
for check in planted.CHECKS:
    print(f'{check:42s}' + ''.join(f'{tables[bug][check]!s:>13s}' for bug in (None, *planted.BUGS)))

Localization compares intermediate arrays with the correct code: the PSDF bug leaves the PTDF untouched, the
float32 bug moves everything by about $10^{-8}$, the slack bug changes a whole PTDF column.

In [ ]:
for bug in planted.BUGS:
    print(f'{bug:12s}', {k: f'{v:.1e}' for k, v in planted.localize(data, bug).items()})

## A bridge under `jit`

The same jitted N-1 function run three ways. Plain: NaN in the row of the bridge. `jax_debug_nans`: an exception.
A success flag: the batched solver's answer.

In [ ]:
flows = bridge_debug.run_plain()
print('rows with NaN:', np.flatnonzero(np.isnan(flows).any(axis=1)))
print('jax_debug_nans ->', bridge_debug.run_debug_nans())
_, ok = bridge_debug.run_flagged()
print('success flags  ->', ok)

## Recompilation and silent no-ops

In [ ]:
for fresh in (True, False):
    traces, seconds = recompile.run(5, fresh, batch=64, n_branches=500)
    label = 'new lambda each call' if fresh else 'module-level function'
    print(f'{label:22s}: {traces} traces, ms per call', [round(1000 * s, 1) for s in seconds])
print(pydantic_extras.LooseParameters(enable_bb_outage=True).model_dump())
for cls in (pydantic_extras.LooseParameters, pydantic_extras.StrictParameters):
    print(cls.__name__, '->', pydantic_extras.extra_policy(cls))

## Hypothesis shrinks the off-by-one in the BSDF busbar mask

In [ ]:
fr, to, b, p, node, to_b = bsdf_property.smallest_counterexample()
print(f'{len(p)} buses, {len(fr)} branches, split node {node}, moved mask {to_b.tolist()}')
mismatch, residual = bsdf_property.violation((fr, to, b, p, node, to_b), off_by_one=True)
print(f'with the bug: PTDF mismatch {mismatch:.3f}, KCL residual {residual:.3f}')